# 01 — From source to searchable chunks

**10–15 minute lab.** Follow three checkpoints: convert one Markdown source, inspect its
structure, then turn it into small index records. The main path is deterministic and needs
neither PostgreSQL nor Ollama.

**Flow:** source → canonical Markdown → AST → chunks → index concept.


In [ ]:
from pathlib import Path

from raglab import SourceInput
from raglab.chunking import ChunkingConfig, SemanticChunker, SimpleTokenCounter
from raglab.conversion import Converter
from raglab.parsing import MarkdownParser

project_root = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
source_path = project_root / "data/samples/aster_greenhouse_controller_manual.md"


## Checkpoint 1 — Objective: create canonical Markdown

**Run:** convert the controlled source through RAGLab's real `Converter` API.


In [ ]:
converted = Converter().convert(SourceInput.path(source_path, purpose="short-lab"))
print(f"Source: {converted.source_name}")
print(f"Converter: {converted.converter}")
print(f"SHA-256: {converted.content_hash[:12]}…")
print("Preview:", converted.markdown.splitlines()[0])


### What to observe

Expect one stable content hash and a Markdown heading in the preview. Conversion creates a
canonical representation before parsing or chunking.

### Conclusion

A source file is input; canonical Markdown is the normalized document the pipeline can inspect.


## Checkpoint 2 — Objective: reveal document structure

**Run:** parse the canonical Markdown into typed AST blocks with heading context.


In [ ]:
parsed = MarkdownParser().parse(converted)
rows = [
    (block.kind.value, " > ".join(block.heading_path) or "—", block.content[:55])
    for block in parsed.blocks
    if block.kind.value != "heading"
]
print(f"Title: {parsed.title}")
print(f"Blocks: {len(parsed.blocks)}")
for kind, path, preview in rows[:5]:
    print(f"- {kind:9} | {path} | {preview}")


### What to observe

Expect several block types and heading paths. The AST stores meaning that a plain string does
not: paragraphs know which section contains them.

### Conclusion

Structure supplies boundaries and context before token limits are applied.


## Checkpoint 3 — Objective: build the indexable unit

**Run:** chunk the parsed document with the real chunker, then represent each chunk as one
searchable index record.


In [ ]:
chunker = SemanticChunker(
    token_counter=SimpleTokenCounter(),
    config=ChunkingConfig(target_tokens=80, min_tokens=30, max_tokens=120),
)
chunks = chunker.chunk(parsed)
index_records = [
    {
        "id": f"chunk-{chunk.index}",
        "text": chunk.content,
        "embedding_text": chunk.embedding_text,
        "heading": " > ".join(chunk.heading_path) or "—",
    }
    for chunk in chunks
]

query_terms = {"irrigation", "flow"}
matches = [
    record for record in index_records
    if query_terms & set(record["embedding_text"].lower().split())
]
print(f"Chunks ready to index: {len(index_records)}")
for record in matches[:3]:
    print(f"- {record['id']} | {record['heading']} | {record['text'][:70]}")


### What to observe

Expect multiple compact records and at least one irrigation-related match. A production index
adds embeddings and BM25 statistics, but its core job is the same: map a query to chunk IDs.

### Conclusion

The index does not store a magical answer. It makes evidence chunks findable while preserving
their text and structural context.

## Optional appendix — real services and deeper experiments

Use `raglab-ingest` for Ollama + PostgreSQL indexing, and use
`benchmark_ingestion_hyperparameters.ipynb` for chunking/HNSW experiments. They are deliberately
outside this short learning path.
